In [65]:
#!pip install skrebate scikit-learn pandas numpy matplotlib seaborn

$\textbf{Step 0: Importing all necessary libraries}$

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, pickle, os, sys
warnings.filterwarnings('ignore')

#Importing The ReliefF Library to Implement for Phase 2
from skrebate import ReliefF

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

#Importing Libraries for the 5 Classifiers that will be used
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

try:
    from tensorboardX import SummaryWriter
    TB_AVAILABLE = True
except ImportError:
    TB_AVAILABLE = False
    print("TensorBoard not available – install tensorboardX to enable it.")

$\textbf{Step 1: Loading Datasets and Setting the Parameters for Models and}$
$\textbf{ReliefF Algorithm}$

In [67]:
DATASETS = {
    'Dataset1' : {'train': 'Datasets/1/train.csv','test': None},
    'Dataset2' : {'train': 'Datasets/2/train.csv','test': None},
    'Dataset3' : {'train': 'Datasets/3/train.csv','test': None},
    'Dataset4' : {'train': 'Datasets/4/train.csv','test': None},
    'Dataset5' : {'train': 'Datasets/5/train.csv','test': None},
    'Dataset6' : {'train': 'Datasets/6/train.csv','test': None},
    'Dataset7' : {'train': 'Datasets/7/train.csv','test': None},
    'Dataset8' : {'train': 'Datasets/8/train.csv','test': None},
}

LABEL_COL = 'Label'
N_FOLDS = 10
INNER_FOLDS = 3
RANDOM_STATE = 42

# ReliefF Settings
# k = number of top features to keep after ranking
# Try a few values; the best k is selected by inner CV
K_CANDIDATES = [5, 10, 15]  # adjust based on total feature count
RELIEFF_N_NEIGHBORS = 10          # ReliefF neighbourhood size

CHECKPOINT_FILE = 'results_checkpoint.pkl'
EXCEL_FILE      = 'formatted_results.xlsx'
PLOT_DIR        = 'plots'
TB_LOG_DIR      = 'tb_logs'

$\textbf{Step 2: Classifier Definition with Hyperparameter Grids}$

In [68]:
CLASSIFIERS = {
    'SVM': (
            CalibratedClassifierCV(LinearSVC(random_state=RANDOM_STATE, max_iter=2000)),
            {'classifier__estimator__C': [1, 10]}
            ),
    'kNN': (
        KNeighborsClassifier(),
        {
            'classifier__n_neighbors': [3, 5, 11, 21],
            'classifier__weights':     ['uniform', 'distance'],
        }
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'classifier__max_depth':        [5, 10, 20, None],
            'classifier__min_samples_split': [2, 5, 10],
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
        {
            'classifier__n_estimators': [50, 100, 200],
            'classifier__max_depth':    [5, 10, None],
        }
    ),
    'MLP': (
        MLPClassifier(max_iter=500, random_state=RANDOM_STATE, early_stopping=True),
        {
            'classifier__hidden_layer_sizes': [(64,), (128,)],
            'classifier__alpha':              [1e-4, 1e-3],
        }
    ),
}

$\textbf{Step 4: Helper Functions}$

In [69]:
def load_dataset(path, label_col=LABEL_COL):
    df = pd.read_csv(path)
    X = df.drop(columns=[label_col]).values.astype(np.float64) #Dropping the Label Column and converts to float64 Numpy array
    y_raw = df[label_col].values
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    print(f'  Loaded: {X.shape[0]} samples, {X.shape[1]} features, '
          f'{len(np.unique(y))} classes')
    return X, y, le

def apply_relieff(X_train, y_train, k, n_neighbors=RELIEFF_N_NEIGHBORS, max_samples=500):
    # Subsample for ReliefF only — avoids distance matrix memory error
    if X_train.shape[0] > max_samples:
        idx = np.random.RandomState(RANDOM_STATE).choice(
            X_train.shape[0], max_samples, replace=False)
        X_relief = X_train[idx]
        y_relief = y_train[idx]
    else:
        X_relief = X_train
        y_relief = y_train

    imputer = SimpleImputer(strategy='mean')
    X_relief_imp = imputer.fit_transform(X_relief)

    selector = ReliefF(n_features_to_select=k, n_neighbors=n_neighbors)
    selector.fit(X_relief_imp, y_relief)
    top_idx = np.argsort(selector.feature_importances_)[::-1][:k]
    return X_train[:, top_idx], selector.feature_importances_, top_idx

def build_pipeline(clf):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('classifier', clf),
    ])

def evaluate_classifier(clf, param_grid, X_train, y_train, X_test, y_test):
    pipe = build_pipeline(clf)
    inner_cv = StratifiedKFold(n_splits=INNER_FOLDS, shuffle=True,
                                random_state = RANDOM_STATE)
    gs = GridSearchCV(pipe, param_grid, cv=inner_cv,
                      scoring='f1_macro', n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    y_pred = gs.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division = 0)
    best_params = gs.best_params_
    return acc, f1, best_params

def print_fold_results(fold_idx, n_folds, ds_name, all_results, clf_names, best_k):
    print(f"\n  ── Fold {fold_idx + 1}/{n_folds} Results ({ds_name}) | best_k={best_k} ──")
    
    rows = []
    for phase_name, classifiers in all_results[ds_name].items():
        for clf_name, metrics in classifiers.items():
            if len(metrics['acc']) == 0:
                continue
            rows.append({
                'Phase':      'Baseline' if phase_name == 'baseline' else 'ReliefF',
                'Classifier': clf_name,
                'Accuracy':  f"{metrics['acc'][fold_idx]:.4f}",
                'Macro-F1':  f"{metrics['f1'][fold_idx]:.4f}",
                'Best Params': metrics['params'][-1],
            })
    
    df = pd.DataFrame(rows)
    pd.set_option('display.max_colwidth', 60)
    display(df)
    print()


$\textbf{Step 5: Main Experiment Loop}$

In [70]:
def run_experiments(datasets, classifiers):
    all_results    = {}
    relieff_scores = {}
    outer_cv       = StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                     random_state=RANDOM_STATE)

    tb_writer = SummaryWriter(log_dir=TB_LOG_DIR) if TB_AVAILABLE else None
    if tb_writer:
        os.makedirs(TB_LOG_DIR, exist_ok=True)

    total_start = time.time()

    for ds_name, paths in datasets.items():
        print(f"\n{'='*60}")
        print(f"  Dataset: {ds_name}")
        print(f"{'='*60}")
        X, y,_ = load_dataset(paths['train'])

        all_results[ds_name] = {
            'baseline': {c: {'acc': [], 'f1': [], 'params': []}
                         for c in classifiers},
            'relieff':  {c: {'acc': [], 'f1': [], 'params': []}
                         for c in classifiers},
        }
        fold_feature_scores = []

        for fold_idx, (train_idx, test_idx) in \
                enumerate(outer_cv.split(X, y)):

            print(f"\n  ── Fold {fold_idx+1}/{N_FOLDS} ──")
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]

            # ── Phase 1 : Baseline ──────────────────────────────
            print("  [Baseline]")
            for clf_name, (clf, grid) in classifiers.items():
                t0          = time.time()
                acc, f1, params = evaluate_classifier(
                    clf, grid, X_tr, y_tr, X_te, y_te)
                elapsed     = time.time() - t0

                all_results[ds_name]['baseline'][clf_name]['acc'].append(acc)
                all_results[ds_name]['baseline'][clf_name]['f1'].append(f1)
                all_results[ds_name]['baseline'][clf_name]['params'].append(params)

                if tb_writer:
                    tb_writer.add_scalar(
                        f'{ds_name}/baseline/{clf_name}/accuracy', acc,  fold_idx)
                    tb_writer.add_scalar(
                        f'{ds_name}/baseline/{clf_name}/f1',       f1,   fold_idx)

                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}"
                      f"  ({elapsed:.1f}s)")

            # ── Phase 2 : ReliefF ───────────────────────────────
            print("  [ReliefF] ranking features …", end=" ", flush=True)
            t0 = time.time()
            _, importances, _ = apply_relieff(X_tr, y_tr, int(max(K_CANDIDATES)))
            print(f"{time.time()-t0:.1f}s")

            # Pick best k with a fast RF proxy in inner CV
            best_k, best_score = K_CANDIDATES[0], -np.inf
            for k in K_CANDIDATES:
                if k > X_tr.shape[1]:
                    continue
                top_idx  = np.argsort(importances)[::-1][:k]
                Xtr_k    = X_tr[:, top_idx]
                inner    = StratifiedKFold(n_splits=3, shuffle=True,
                                           random_state=0)
                scores   = []
                for itr, ite in inner.split(Xtr_k, y_tr):
                    rf = RandomForestClassifier(n_estimators=50,
                                                random_state=RANDOM_STATE,
                                                n_jobs=-1)
                    rf.fit(Xtr_k[itr], y_tr[itr])
                    scores.append(f1_score(y_tr[ite],
                                           rf.predict(Xtr_k[ite]),
                                           average='macro', zero_division=0))
                if np.mean(scores) > best_score:
                    best_score = np.mean(scores)
                    best_k     = k

            top_final = np.argsort(importances)[::-1][:best_k]
            X_tr_r    = X_tr[:, top_final]
            X_te_r    = X_te[:, top_final]
            fold_feature_scores.append(importances)
            print(f"  [ReliefF] best_k={best_k}")

            for clf_name, (clf, grid) in classifiers.items():
                t0          = time.time()
                acc, f1, params = evaluate_classifier(
                    clf, grid, X_tr_r, y_tr, X_te_r, y_te)
                elapsed     = time.time() - t0

                all_results[ds_name]['relieff'][clf_name]['acc'].append(acc)
                all_results[ds_name]['relieff'][clf_name]['f1'].append(f1)
                all_results[ds_name]['relieff'][clf_name]['params'].append(params)

                if tb_writer:
                    tb_writer.add_scalar(
                        f'{ds_name}/relieff/{clf_name}/accuracy', acc,  fold_idx)
                    tb_writer.add_scalar(
                        f'{ds_name}/relieff/{clf_name}/f1',       f1,   fold_idx)

                print(f"    {clf_name:15s}  acc={acc:.4f}  f1={f1:.4f}"
                      f"  ({elapsed:.1f}s)")

        relieff_scores[ds_name] = np.mean(fold_feature_scores, axis=0)

        # Save checkpoint after every dataset (safe if interrupted)
        with open(CHECKPOINT_FILE, 'wb') as f:
            pickle.dump({'all_results':    all_results,
                         'relieff_scores': relieff_scores}, f)
        print(f"\n  ✅  Checkpoint saved after {ds_name}")

    if tb_writer:
        tb_writer.flush()
        tb_writer.close()

    elapsed_total = time.time() - total_start
    print(f"\n{'='*60}")
    print(f"  All experiments done in {elapsed_total/60:.1f} min")
    print(f"{'='*60}\n")
    return all_results, relieff_scores

In [71]:
def make_plots(all_results, relieff_scores):
    
    os.makedirs(PLOT_DIR, exist_ok=True)
    CLF_NAMES  = ['SVM', 'kNN', 'Decision Tree', 'Random Forest', 'MLP']
    DS_NAMES   = sorted(all_results.keys())
    PALETTE    = sns.color_palette('tab10', len(CLF_NAMES))
    CLF_COLOR  = dict(zip(CLF_NAMES, PALETTE))
    PHASES     = ['baseline', 'relieff']
    PHASE_LBLS = {'baseline': 'Baseline', 'relieff': 'ReliefF'}

    def ms(ds, phase, clf, metric):
        v = all_results[ds][phase][clf][metric]
        return np.mean(v), np.std(v)

    # ── Plot 1 : Bar chart per dataset ─────────────────────────
    for ds in DS_NAMES:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
        fig.suptitle(f'{ds}  –  Mean ± Std  (10-fold CV)',
                     fontsize=13, fontweight='bold')
        for ax, metric in zip(axes, ['acc', 'f1']):
            x     = np.arange(len(CLF_NAMES))
            width = 0.35
            for i, phase in enumerate(PHASES):
                means = [ms(ds, phase, c, metric)[0] for c in CLF_NAMES]
                stds  = [ms(ds, phase, c, metric)[1] for c in CLF_NAMES]
                bars  = ax.bar(x + i*width - width/2, means, width,
                               label=PHASE_LBLS[phase],
                               color=['#4C72B0', '#DD8452'][i],
                               yerr=stds, capsize=4, alpha=0.87,
                               error_kw={'elinewidth': 1.2})
                for bar, m in zip(bars, means):
                    ax.text(bar.get_x() + bar.get_width()/2,
                            bar.get_height() + 0.005,
                            f'{m:.3f}', ha='center', va='bottom', fontsize=7)
            ax.set_xticks(x)
            ax.set_xticklabels(CLF_NAMES, rotation=20, ha='right', fontsize=9)
            ax.set_ylabel('Accuracy' if metric == 'acc' else 'Macro-F1')
            ax.set_ylim(0, 1.08)
            ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
            ax.legend(fontsize=9)
            ax.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'{ds}_bar.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()

    # ── Plot 2 : Fold-by-fold F1 line ──────────────────────────
    for ds in DS_NAMES:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.set_title(f'{ds}  –  F1 per fold (Baseline)',
                     fontsize=12, fontweight='bold')
        folds = np.arange(1, N_FOLDS + 1)
        for clf in CLF_NAMES:
            vals = all_results[ds]['baseline'][clf]['f1']
            ax.plot(folds, vals, marker='o', label=clf,
                    color=CLF_COLOR[clf], linewidth=1.8, markersize=5)
        ax.set_xlabel('Fold')
        ax.set_ylabel('Macro-F1')
        ax.set_xticks(folds)
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=9, ncol=2)
        ax.grid(linestyle='--', alpha=0.45)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'{ds}_folds_line.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()

    # ── Plot 3 : ΔF1 heatmap ───────────────────────────────────
    delta = np.zeros((len(DS_NAMES), len(CLF_NAMES)))
    for i, ds in enumerate(DS_NAMES):
        for j, clf in enumerate(CLF_NAMES):
            delta[i, j] = (np.mean(all_results[ds]['relieff'][clf]['f1']) -
                           np.mean(all_results[ds]['baseline'][clf]['f1']))
    fig, ax = plt.subplots(figsize=(10, max(3, len(DS_NAMES)*0.7)))
    sns.heatmap(delta, annot=True, fmt='.3f',
                xticklabels=CLF_NAMES, yticklabels=DS_NAMES,
                cmap='RdYlGn', center=0,
                linewidths=0.5, linecolor='grey', ax=ax)
    ax.set_title('ΔF1  (ReliefF − Baseline)  per Dataset × Classifier',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, 'delta_f1_heatmap.png'),
                dpi=150, bbox_inches='tight')
    plt.close()

    # ── Plot 4 : ReliefF feature importance ────────────────────
    for ds in DS_NAMES:
        scores = relieff_scores.get(ds)
        if scores is None:
            continue
        top_n   = min(20, len(scores))
        top_idx = np.argsort(scores)[::-1][:top_n]
        top_sc  = scores[top_idx]
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.barh(np.arange(top_n)[::-1], top_sc,
                color=sns.color_palette('viridis', top_n))
        ax.set_yticks(np.arange(top_n)[::-1])
        ax.set_yticklabels([f'F{i}' for i in top_idx], fontsize=8)
        ax.set_xlabel('Mean ReliefF Score')
        ax.set_title(f'{ds}  –  Top-{top_n} Feature Importances (ReliefF)',
                     fontsize=12, fontweight='bold')
        ax.grid(axis='x', linestyle='--', alpha=0.45)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOT_DIR, f'{ds}_relieff_importance.png'),
                    dpi=150, bbox_inches='tight')
        plt.close()

    print(f"✅  All plots saved to '{PLOT_DIR}/'")


In [72]:
def export_excel(all_results):
    model_map = {
        'SVM': 'SVM', 'kNN': 'KNN',
        'Decision Tree': 'DT', 'Random Forest': 'RF', 'MLP': 'MLP'
    }

    def build_sheet(phase):
        all_rows = []
        for i, ds in enumerate(sorted(all_results.keys()), 1):
            block   = all_results[ds][phase]
            cols    = []
            for m in model_map.values():
                cols += [f'{m}-Accuracy', f'{m}-F1']
            cols += [f'{m} Parameters' for m in model_map.values()]

            all_rows.append([f'Data {i}'] + [''] * len(cols))
            all_rows.append([''] + cols)

            n_folds   = len(next(iter(block.values()))['acc'])
            fold_data = []
            for fold in range(n_folds):
                row, num_row = [f'Fold {fold+1}'], []
                for model in model_map:
                    acc = block[model]['acc'][fold]
                    f1  = block[model]['f1'][fold]
                    row += [round(acc, 4), round(f1, 4)]
                    num_row += [acc, f1]
                for model in model_map:
                    row.append(str(block[model]['params'][fold]))
                all_rows.append(row)
                fold_data.append(num_row)

            arr      = np.array(fold_data)
            means    = np.mean(arr, axis=0)
            stds     = np.std(arr,  axis=0)
            mean_std = [f'{m:.4f} ± {s:.4f}' for m, s in zip(means, stds)]
            all_rows.append(['Mean ± Std'] + mean_std + [''] * len(model_map))
            all_rows.append([''] * (len(cols) + 1))

        return pd.DataFrame(all_rows)

    with pd.ExcelWriter(EXCEL_FILE, engine='openpyxl') as writer:
        build_sheet('baseline').to_excel(
            writer, sheet_name='Baseline', index=False, header=False)
        build_sheet('relieff').to_excel(
            writer, sheet_name='ReliefF',  index=False, header=False)

    print(f"✅  Excel saved → '{EXCEL_FILE}'")

In [ ]:
datasets    = DATASETS
classifiers = CLASSIFIERS

all_results, relieff_scores = run_experiments(datasets, classifiers)

make_plots(all_results, relieff_scores)
export_excel(all_results)

print("\nDone! Files created:")
print(f"  • {CHECKPOINT_FILE}")
print(f"  • {EXCEL_FILE}")
print(f"  • {PLOT_DIR}/  (all plots)")
if TB_AVAILABLE:
    print(f"  • {TB_LOG_DIR}/  →  run: tensorboard --logdir {TB_LOG_DIR}")


  Dataset: Dataset1
  Loaded: 9583 samples, 19 features, 4 classes

  ── Fold 1/10 ──
  [Baseline]
    SVM              acc=0.9864  f1=0.7372  (4.0s)
    kNN              acc=0.9906  f1=0.7368  (0.9s)
    Decision Tree    acc=1.0000  f1=1.0000  (0.7s)
    Random Forest    acc=1.0000  f1=1.0000  (6.2s)
    MLP              acc=0.9823  f1=0.7329  (8.4s)
  [ReliefF] ranking features … 1.1s
  [ReliefF] best_k=5
    SVM              acc=0.8467  f1=0.4786  (0.9s)
    kNN              acc=0.9990  f1=0.9990  (0.6s)
    Decision Tree    acc=1.0000  f1=1.0000  (0.2s)
    Random Forest    acc=1.0000  f1=1.0000  (4.6s)
    MLP              acc=0.8394  f1=0.4419  (2.8s)

  ── Fold 2/10 ──
  [Baseline]
    SVM              acc=0.9875  f1=0.7341  (2.1s)
    kNN              acc=0.9917  f1=0.7344  (1.3s)
    Decision Tree    acc=1.0000  f1=1.0000  (0.6s)
    Random Forest    acc=1.0000  f1=1.0000  (8.5s)
    MLP              acc=0.9854  f1=0.7322  (9.3s)
  [ReliefF] ranking features … 1.0s
  [ReliefF